In [ ]:
import tensorflow as tf

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Show errors only.

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    print("Available GPUs:")
    for gpu in gpus:
        print(gpu)
else:
    print("No GPU found.")

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth set for GPUs.")
    except RuntimeError as e:
        print(e)

In [ ]:
physical_devices = tf.config.list_physical_devices("GPU")
tf.config.experimental.set_memory_growth(physical_devices[0], True)

In [ ]:
import tensorflow as tf

IMG_SIZE = (100, 100)
BATCH_SIZE = 32

def get_dataset(image_dir):
    dataset = tf.keras.preprocessing.image_dataset_from_directory(
        image_dir,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
    return dataset

dataset = get_dataset('/kaggle/input/lung-and-colon-cancer-histopathological-images/lung_colon_image_set/lung_image_sets')

dataset_size = tf.data.experimental.cardinality(dataset).numpy()

train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = int(.15 * dataset_size)

train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size).take(val_size)
test_ds = dataset.skip(train_size + val_size)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomWidth(0.2),
    tf.keras.layers.RandomHeight(0.2)
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
train_ds = train_ds.cache()
val_ds = val_ds.cache()
test_ds = test_ds.cache()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, regularizers

model = keras.Sequential([
    keras.Input((100, 100, 3)),

    layers.Conv2D(64, 3, activation='relu', padding='same', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.3),

    layers.Conv2D(128, 3, activation='relu', padding='same', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.3),

    layers.Conv2D(256, 3, activation='relu', padding='same', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.4),

    layers.Conv2D(512, 3, activation='relu', padding='same', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.5),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),

    layers.Dropout(0.5),

    layers.Dense(3, activation='softmax')
])

In [ ]:
model.summary()

In [ ]:
from keras.optimizers import Adam

optimizer = Adam()

In [ ]:
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(train_ds, validation_data=val_ds, verbose=2, epochs=50)

In [ ]:
model.evaluate(test_ds, verbose=1)

In [ ]:
model.save("lung_cancer_model.h5")